# SQL 6: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-06-joins.ipynb)

Real databases split information across tables. `customers` knows each customer's name, `orders` knows each
order's amount, and the two are linked by `customer_id`. A **join** combines the tables by matching that key, so a
single query can answer questions that span both.

This notebook explains each join type, the rules that decide how many rows come back, and the mistakes that
most often produce wrong totals. It ends with practice questions and their answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customer_tags AS
SELECT * FROM VALUES (1, 'vip'), (1, 'eu'), (2, 'vip') AS customer_tags(customer_id, tag)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW staff AS
SELECT * FROM VALUES
  (1, 'Alice', NULL),
  (2, 'Bob',   1),
  (3, 'Carol', 1),
  (4, 'Dave',  2),
  (5, 'Eve',   2)
AS staff(emp_id, name, manager_id)
""")

### The data

- `customers`: 8 rows, one per customer. `customer_id` is unique here.
- `orders`: 14 rows for customers 1 to 6. Customers 7 and 8 have no orders.
- `customer_tags`: 3 rows. Customer 1 has two tags (`vip`, `eu`), customer 2 has one (`vip`), the rest have none.
- `staff`: 5 employees. `manager_id` points at another row's `emp_id`; Alice has no manager.

Each table's **grain** is what one row represents: one customer, one order, one tag on a customer, one employee.
Knowing the grain of each side is how you predict the row count of a join.

In [ ]:
for table, rows in [("customers", 8), ("orders", 14), ("customer_tags", 3), ("staff", 5)]:
    assert spark.table(table).count() == rows, table

## 1. The join condition

Every join needs a **condition** that says which rows match. It is written after `ON`:

```sql
SELECT ...
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id;
```

Read it as: for each order, find the customer whose `customer_id` is the same. Table aliases (`o`, `c`) keep
the column references short and tell you which table each column comes from.

The column you match on is the **join key**. The condition is a **predicate**: an expression that is true, false or
unknown for each pair of rows.

### Where you'll use it

- **Keys are the contract:** most joins in a warehouse use a surrogate key from a dimension table. A wrong key condition is the most common source of duplicate rows.

### A join is every pair, filtered

The simplest way to reason about a join: imagine every pairing of a left row with a right row, then keep the pairs for
which the condition is true. Engines use faster algorithms, but the result is always the same as this picture.

Two tiny tables make it concrete. `left_keys` holds ids 1 and 2; `right_keys` holds ids 2 and 3.

| left id | right id | `l.id = r.id` |
|---|---|---|
| 1 | 2 | false |
| 1 | 3 | false |
| 2 | 2 | **true** |
| 2 | 3 | false |

Four pairs, one of which satisfies the condition. An inner join returns exactly that pair.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW left_keys AS SELECT * FROM VALUES (1), (2) AS left_keys(id)")
spark.sql("CREATE OR REPLACE TEMP VIEW right_keys AS SELECT * FROM VALUES (2), (3) AS right_keys(id)")

all_pairs = "SELECT l.id AS left_id, r.id AS right_id FROM left_keys l CROSS JOIN right_keys r ORDER BY l.id, r.id"
matched = "SELECT l.id AS left_id, r.id AS right_id FROM left_keys l JOIN right_keys r ON l.id = r.id"
spark.sql(all_pairs).show()

assert spark.sql(all_pairs).collect() == [(1, 2), (1, 3), (2, 2), (2, 3)]
assert spark.sql(matched).collect() == [(2, 2)]

### Cardinality: how many rows come back

The pairing picture predicts the row count. Each left row appears once for every right row it matches.

- **One-to-one:** each key appears at most once on each side. The result has at most as many rows as the smaller side.
- **One-to-many:** the key is unique on one side only, such as one customer to many orders. The result has one row
  per row of the "many" side that finds a match.
- **Many-to-many:** the key repeats on both sides. Each combination is returned, so rows multiply (section 8).

`orders` to `customers` is many-to-one: every order has exactly one customer, so the join returns 14 rows, one per order.

In [ ]:
q = "SELECT COUNT(*) FROM orders o JOIN customers c ON c.customer_id = o.customer_id"
assert spark.sql("SELECT COUNT(DISTINCT customer_id) = COUNT(*) FROM customers").collect() == [(True,)]
assert spark.sql(q).collect() == [(14,)]

## 2. Inner join

`INNER JOIN` (or just `JOIN`) keeps only the rows that match in **both** tables. An order whose customer does not exist, or a customer with no orders, disappears from the result.

A row whose join key is NULL also disappears: `NULL = x` is unknown, never true, so the pair is not kept.

### Where you'll use it

- **Requiring a match:** orders joined to customers, so orders with an unknown customer drop out. Good for reports that must be complete.

In [ ]:
q = """
SELECT o.order_id, c.name, o.amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
LIMIT 5
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, "Acme Corp", 100), (2, "Acme Corp", 200), (3, "Acme Corp", 150), (4, "Beta LLC", 300), (5, "Beta LLC", 250),
]

Every one of the 14 orders matches a customer, so the join returns 14 rows. Customers 7 and 8 have no orders,
so they are not in the result. The check below proves both statements.

In [ ]:
joined = spark.sql("""
SELECT o.order_id, c.customer_id
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
""")
assert joined.count() == 14
assert sorted(r.customer_id for r in joined.select("customer_id").distinct().collect()) == [1, 2, 3, 4, 5, 6]

## 3. Left join

`LEFT JOIN` keeps **every row of the left table**, even when nothing matches on the right. The right-hand
columns are filled with `NULL` for those rows. Use it when the left table is the list you want to report on.

Row count: every matched pair, plus one row for each left row that matched nothing. Customers 1 to 6 match 14 orders
in total, and customers 7 and 8 add one row each, so `customers LEFT JOIN orders` returns 16 rows.

The classic use is finding the unmatched rows: a `LEFT JOIN` followed by `WHERE right_key IS NULL`.

### Where you'll use it

- **Keeping every fact:** revenue per campaign, including campaigns with no sales, which is the default join for reporting.

In [ ]:
q = """
SELECT c.customer_id, c.name, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE c.customer_id >= 7
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(7, "Gamma SA", None), (8, "Hotel BV", None)]
assert spark.sql("SELECT COUNT(*) FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id").collect() == [(16,)]

### Finding unmatched rows

After a `LEFT JOIN`, a NULL in a right-hand column that can never be NULL in its own table (here the key `order_id`)
marks a left row with no match. Filtering on it lists the customers who never ordered.

In [ ]:
q = """
SELECT c.customer_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(7,), (8,)]

## 4. Right join and full outer join

`RIGHT JOIN` is the mirror image of `LEFT JOIN`: it keeps every row of the right table. `FULL OUTER JOIN` keeps
every row of **both** tables. Unmatched rows from either side get `NULL`s for the other side's columns.

The small example below uses `left_keys` (1, 2) and `right_keys` (2, 3). Id 2 matches. Id 1 exists only on the left,
so its right column is NULL. Id 3 exists only on the right, so its left column is NULL. Sorting by
`COALESCE(l.id, r.id)` uses whichever side is present.

### Where you'll use it

- **Reconciliation:** a full outer join between two systems shows what exists only in each one.

In [ ]:
q = """
SELECT l.id AS left_id, r.id AS right_id
FROM left_keys l
FULL OUTER JOIN right_keys r ON l.id = r.id
ORDER BY COALESCE(l.id, r.id)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, None), (2, 2), (None, 3)]

### Right join

`RIGHT JOIN` keeps every row of the right-hand table. It is the same as a `LEFT JOIN` with the tables swapped, so most
people write `LEFT JOIN` and reorder the tables. Here the right table has an id that the left table lacks. The second
query swaps the tables and uses `LEFT JOIN`; the rows are identical.

In [ ]:
right = """
SELECT l.id AS left_id, r.id AS right_id
FROM left_keys l
RIGHT JOIN right_keys r ON l.id = r.id
ORDER BY right_id
"""
swapped = """
SELECT l.id AS left_id, r.id AS right_id
FROM right_keys r
LEFT JOIN left_keys l ON l.id = r.id
ORDER BY right_id
"""
spark.sql(right).show()
assert spark.sql(right).collect() == [(2, 2), (None, 3)]
assert spark.sql(swapped).collect() == spark.sql(right).collect()

## 5. Cross join

A `CROSS JOIN` pairs every row of the left table with every row of the right table. There is no condition. The
result has `left rows × right rows` rows, so use it deliberately, for example to build a grid of combinations.

There are 4 distinct regions and 3 distinct statuses, so the grid below has 4 × 3 = 12 rows.

### Where you'll use it

- **Building a grid:** every store paired with every day, the first step of a calendar report.

In [ ]:
q = """
SELECT r.region, s.status
FROM (SELECT DISTINCT region FROM customers) r
CROSS JOIN (SELECT DISTINCT status FROM orders) s
ORDER BY r.region, s.status
"""
spark.sql(q).show(12)
rows = spark.sql(q).collect()
assert len(rows) == 4 * 3
assert rows[:3] == [("East", "cancelled"), ("East", "completed"), ("East", "pending")]

## 6. Semi join and anti join

Two joins return only columns from the left table:

- `LEFT SEMI JOIN` keeps each left row that has **at least one** match. It is the same as `WHERE EXISTS`.
- `LEFT ANTI JOIN` keeps each left row that has **no** match. It is the same as `WHERE NOT EXISTS`.

Neither can duplicate a row, which makes them safer than an inner join when you only need to test for a match. A semi
join answers "is there any match?"; it stops caring after the first one.

### Where you'll use it

- **Filtering by membership:** users who bought in the last 30 days (semi), and users who never bought (anti) for a win-back campaign.

In [ ]:
semi = """
SELECT c.name
FROM customers c
LEFT SEMI JOIN orders o ON o.customer_id = c.customer_id
ORDER BY c.name
"""
anti = """
SELECT c.name
FROM customers c
LEFT ANTI JOIN orders o ON o.customer_id = c.customer_id
ORDER BY c.name
"""
spark.sql(semi).show()
spark.sql(anti).show()

assert spark.sql(semi).collect() == [
    ("Acme Corp",), ("Beta LLC",), ("Cyan Inc",), ("Delta Co",), ("Echo Ltd",), ("Foxtrot GmbH",),
]
assert spark.sql(anti).collect() == [("Gamma SA",), ("Hotel BV",)]

### Semi join versus inner join

The semi join above returns 6 rows, one per customer with orders. An inner join of the same tables returns 14 rows,
because each customer is repeated once per order. The `EXISTS` form gives exactly the semi join's rows.

In [ ]:
inner_count = spark.sql("SELECT COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id").collect()
exists = """
SELECT c.name FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
ORDER BY c.name
"""
assert inner_count == [(14,)]
assert spark.sql(exists).collect() == spark.sql(semi).collect()

## 7. Conditions in ON versus WHERE

This is the mistake that trips up most people with outer joins.

- A condition on the right-hand table in `ON` decides which right rows can match. Left rows without a match are
  still kept.
- The same condition in `WHERE` runs after the join. It removes the `NULL` rows, which quietly turns a
  `LEFT JOIN` into an inner join.

Work it out for customer 7. With the status test in `ON`, no order matches, so the join keeps customer 7 with NULL
order columns, and `COUNT(o.order_id)` is 0. With the test in `WHERE`, the join first produces that NULL row, then
`WHERE NULL = 'completed'` is unknown, and the row is removed before grouping.

The two queries below count completed orders per customer. Only the first keeps customers 7 and 8.

### Where you'll use it

- **Outer joins in practice:** a filter on the right table belongs in `ON`, or the left join quietly becomes an inner join.

In [ ]:
in_on = """
SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
GROUP BY c.customer_id
ORDER BY c.customer_id
"""
in_where = """
SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.customer_id
ORDER BY c.customer_id
"""
spark.sql(in_on).show()
spark.sql(in_where).show()

assert spark.sql(in_on).collect() == [(1, 3), (2, 2), (3, 2), (4, 1), (5, 2), (6, 1), (7, 0), (8, 0)]
assert spark.sql(in_where).collect() == [(1, 3), (2, 2), (3, 2), (4, 1), (5, 2), (6, 1)]

### A left-table condition in ON does not filter

The reverse mistake: a condition on the **left** table placed in `ON` does not remove left rows, because a `LEFT JOIN`
keeps every left row regardless. It only stops those rows from matching. Below, `c.region = 'North'` in `ON` still
returns all 8 customers; only the two North customers get their orders (3 each), and the other 6 get one NULL row each.
That is 6 + 6 = 12 rows. To keep only North customers, put the test in `WHERE`.

In [ ]:
q = """
SELECT c.customer_id, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND c.region = 'North'
"""
result = spark.sql(q)
assert result.count() == 12
assert result.select("customer_id").distinct().count() == 8
assert result.where("order_id IS NOT NULL").count() == 6

## 8. Row multiplication: why totals can be wrong

A join returns one row for every match. When one customer matches several rows in another table, the customer's
orders are repeated, and a `SUM` counts them more than once.

`customer_tags` has two tags for customer 1. Joining orders to it repeats each of customer 1's orders twice. The
correct total of completed orders for customers 1 and 2 is 1,150, but the naive join reports 1,600.

By hand: customer 1's completed orders are 100 + 200 + 150 = 450, and customer 2's are 300 + 400 = 700, so the
correct total is 1,150. In the join, customer 1's three orders each appear twice (once per tag), adding 900 instead
of 450: 900 + 700 = 1,600. The row count shows it too: 5 completed orders become 8 joined rows.

The fix is to ask a membership question instead of joining: a semi join keeps each order once.

### Where you'll use it

- **Wrong totals:** joining orders to a many-row table and summing revenue counts each order once per match. Checking row counts before and after a join catches it.

In [ ]:
naive = """
SELECT SUM(o.amount) AS naive_total, COUNT(*) AS joined_rows
FROM orders o
JOIN customer_tags t ON t.customer_id = o.customer_id
WHERE o.status = 'completed'
"""
correct = """
SELECT SUM(o.amount) AS correct_total, COUNT(*) AS order_rows
FROM orders o
LEFT SEMI JOIN customer_tags t ON t.customer_id = o.customer_id
WHERE o.status = 'completed'
"""
spark.sql(naive).show()
spark.sql(correct).show()

assert spark.sql(naive).collect() == [(1600, 8)]
assert spark.sql(correct).collect() == [(1150, 5)]

### Fix by changing the grain first

If you need a column from the other table, reduce it to one row per key before joining. `customer_tags` has the grain
"one tag per customer"; `SELECT DISTINCT customer_id` turns it into "one row per customer", and the join becomes
many-to-one again.

In [ ]:
q = """
SELECT SUM(o.amount) AS total
FROM orders o
JOIN (SELECT DISTINCT customer_id FROM customer_tags) t ON t.customer_id = o.customer_id
WHERE o.status = 'completed'
"""
assert spark.sql(q).collect() == [(1150,)]

## 9. Self join

A table can join to itself if each side gets its own alias. The `staff` table records each employee's manager, so
joining it to itself shows each employee next to their manager. `LEFT JOIN` keeps Alice, who has no manager.

Read `e` as "the employee row" and `m` as "the manager row". The condition `m.emp_id = e.manager_id` finds, for each
employee, the row whose id is that employee's manager id.

### Where you'll use it

- **Hierarchies:** each employee joined to their manager in the same table.
- **Pairing events:** matching a login to the next logout for the same user.

In [ ]:
q = """
SELECT e.name AS employee, m.name AS manager
FROM staff e
LEFT JOIN staff m ON m.emp_id = e.manager_id
ORDER BY e.emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Alice", None), ("Bob", "Alice"), ("Carol", "Alice"), ("Dave", "Bob"), ("Eve", "Bob"),
]

### Pairs within the same table

To list pairs of employees who share a manager, join `staff` to itself on `manager_id`. Without more care, each pair
appears twice (Bob with Carol, and Carol with Bob) and every employee pairs with themselves. The extra condition
`a.emp_id < b.emp_id` keeps each pair once. Alice's NULL `manager_id` matches nothing, so she is in no pair.

In [ ]:
q = """
SELECT a.name AS first, b.name AS second
FROM staff a
JOIN staff b ON a.manager_id = b.manager_id AND a.emp_id < b.emp_id
ORDER BY a.emp_id, b.emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Bob", "Carol"), ("Dave", "Eve")]

## Interview notes

- **Join cardinality.** Before a join, know whether each key is unique on each side. One-to-one and one-to-many joins
  keep the "one" side's values once; many-to-many multiplies them.
- **Row-count bounds.** An inner join can return anything from 0 rows up to left × right. A left join returns at least
  as many rows as the left table, and more when keys repeat on the right.
- **NULL keys never match** in an equality join. Rows with a NULL key drop out of inner joins and get NULLs in outer joins.
- **ON versus WHERE.** In an outer join, a filter on the optional side belongs in `ON`; a filter on the preserved side
  belongs in `WHERE`.
- **Join order and size.** Engines reorder inner joins to reduce work, but outer joins limit reordering. Filter and
  aggregate early so the joined inputs are small.
- **Broadcast the small side.** When one side is small, the engine can send it to every worker and avoid shuffling the
  large side. The modeling chapter shows the `BROADCAST` hint and its plan.
- **Data skew.** A join key with a very frequent value sends most of the work to one worker. Check the top key counts
  before a large join.

## Summary

- A join is every pair of rows filtered by the condition. Its row count follows from the keys' cardinality.
- An **inner join** keeps matches only. A **left join** keeps every left row.
- **Full outer** keeps both sides; **cross** pairs everything; **semi** and **anti** test for a match.
- Put conditions on the right-hand table in `ON` when unmatched left rows must survive.
- Check the join's row count before you sum anything. One-to-many joins repeat the "one" side's amounts.

## Practice questions

1. List every order with the customer's name, sorted by `order_id`. Show the first three rows.
2. Which customers have no tags? Show their names, sorted by name.
3. For every customer, show the name and the number of tags. Include customers with no tags, sorted by `customer_id`.
4. Show each employee with their manager's name, sorted by `emp_id`. Alice has no manager.
5. For every customer, show `customer_id` and the total amount of completed orders, with 0 for customers who have none.
   Sort by `customer_id`.
6. What is the total amount of completed orders placed by customers tagged `vip`? Make sure no order is counted twice.

## Answers

### Question 1

Inner join orders to customers on the key. Every order has a customer, so nothing is lost.

In [ ]:
q = """
SELECT o.order_id, c.name
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
"""
spark.sql(q).show(3)
assert spark.sql(q).limit(3).collect() == [(1, "Acme Corp"), (2, "Acme Corp"), (3, "Acme Corp")]

### Question 2

An anti join keeps customers with no matching tag row. Only customers 1 and 2 have tags.

In [ ]:
q = """
SELECT c.name
FROM customers c
LEFT ANTI JOIN customer_tags t ON t.customer_id = c.customer_id
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Cyan Inc",), ("Delta Co",), ("Echo Ltd",), ("Foxtrot GmbH",), ("Gamma SA",), ("Hotel BV",)]

### Question 3

A left join keeps every customer; count a right-hand column so unmatched customers count 0.

In [ ]:
q = """
SELECT c.name, COUNT(t.tag) AS tags
FROM customers c
LEFT JOIN customer_tags t ON t.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Acme Corp", 2), ("Beta LLC", 1), ("Cyan Inc", 0), ("Delta Co", 0),
    ("Echo Ltd", 0), ("Foxtrot GmbH", 0), ("Gamma SA", 0), ("Hotel BV", 0),
]

Counting `t.tag` rather than `*` gives 0 for customers without tags, because the `LEFT JOIN` fills their tag with `NULL`.

### Question 4

A self join with a `LEFT JOIN`, so the employee without a manager is kept.

In [ ]:
q = """
SELECT e.name AS employee, m.name AS manager
FROM staff e
LEFT JOIN staff m ON m.emp_id = e.manager_id
ORDER BY e.emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Alice", None), ("Bob", "Alice"), ("Carol", "Alice"), ("Dave", "Bob"), ("Eve", "Bob"),
]

### Question 5

Put the status test in `ON` so customers 3 to 8 without completed orders survive, then turn the empty `SUM` (NULL)
into 0 with `COALESCE`. Customer 5: 500 + 450 = 950.

In [ ]:
q = """
SELECT c.customer_id, COALESCE(SUM(o.amount), 0) AS completed_amount
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
GROUP BY c.customer_id
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 450), (2, 700), (3, 300), (4, 320), (5, 950), (6, 275), (7, 0), (8, 0)]

### Question 6

A semi join asks "does this order's customer have a `vip` tag?" and keeps each order at most once. Customers 1 and 2
are `vip`: 450 + 700 = 1,150.

In [ ]:
q = """
SELECT SUM(o.amount) AS vip_completed
FROM orders o
LEFT SEMI JOIN customer_tags t ON t.customer_id = o.customer_id AND t.tag = 'vip'
WHERE o.status = 'completed'
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1150,)]